# Inheritance and Abstract Classes

CSC-239 · Module 6 · Lesson 2 of 3

You can express a shared interface and call different implementations through it. Now you will give related classes a shared base that can hold state and implemented methods. A concrete subclass will supply the behavior that the base leaves unfinished.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Extend an abstract class while initializing its shared state through super.
- Implement required behavior and combine inherited and overridden behavior through a base-typed reference.


## Why This Matters

A work planner gives each job a name, but reading jobs and building jobs calculate their time differently. A base class can store the name once in the design while each subclass supplies its own calculation.


## Check Your Starting Point

Recall why a constructor initializes each new object, why private fields are accessed through methods, and how a reference’s declared type differs from the actual class. Explain how @Override helps check an intended method implementation.

**My explanation:**


## Concept

### Extend a related class

**Class inheritance** lets a subclass extend a superclass and receive eligible behavior from it. A **superclass**, or base class, supplies the shared definition. A **subclass** specializes that definition. Java classes extend one direct superclass.

A subclass constructor can use a **superclass constructor call**, written super(arguments), to initialize the superclass part of the same object. In this course’s Java 21 runtime, this explicit call belongs first in the constructor.

```java
class NameCard {
    private String name;
    public NameCard(String name) {
        this.name = name;
    }
    public String label() {
        return name;
    }
}
class VisitorCard extends NameCard {
    public VisitorCard(String name) {
        super(name);
    }
}
VisitorCard card = new VisitorCard("Nora");
System.out.println(card.label());
```

This prints `Nora`. VisitorCard inherits the public label method. Its constructor passes the name to NameCard, which initializes its private field. The subclass does not directly access that private field.

Constructors are not inherited. Calling super(name) does not create a separate NameCard object beside the VisitorCard. It initializes the base portion of the one VisitorCard being created. If the base requires a name, the subclass must arrange to supply it.

### Replace inherited behavior, then reuse part of it

**Overriding inherited class behavior** means giving a subclass a matching instance method that replaces the inherited implementation for calls on that subclass object. You already used @Override for an interface operation. The annotation also checks this relationship with a superclass method.

A **superclass method call**, such as super.label(), explicitly calls the accessible superclass implementation. It lets an override build on the earlier behavior:

```java
class NameCard {
    private String name;
    public NameCard(String name) {
        this.name = name;
    }
    public String label() {
        return name;
    }
}
class VisitorCard extends NameCard {
    public VisitorCard(String name) {
        super(name);
    }
    @Override
    public String label() {
        return super.label() + " visitor";
    }
}
NameCard card = new VisitorCard("Sam");
System.out.println(card.label());
```

The output is `Sam visitor`. The declared type allows the label call. The actual VisitorCard object selects the override. Inside that override, super.label() obtains Sam from the base implementation, and the subclass adds its own suffix.

Calling label() again inside this label override would call the override again. It would not mean “call the base version.” Use super.label() when you intend to reuse the base implementation.

### Leave a required operation unfinished in an abstract base

An **abstract class** can contain fields, constructors, and implemented methods, but it cannot be constructed directly. An **abstract class method** declares an operation whose body a concrete subclass must supply. A **concrete class** is complete enough to construct.

```java
abstract class TimedWork {
    public TimedWork() {
    }
    public abstract int minutes();
}
class PackingWork extends TimedWork {
    private int boxes;
    public PackingWork(int boxes) {
        super();
        this.boxes = boxes;
    }
    @Override
    public int minutes() {
        return boxes * 3;
    }
}
TimedWork work = new PackingWork(2);
System.out.println(work.minutes());
```

This prints 6. TimedWork declares the minutes operation with a semicolon and no body. PackingWork implements it. The variable may have the abstract base type even though the actual object must come from a concrete class.

Do not try to construct TimedWork directly. It does not define how to calculate minutes. A subclass that remains missing a required abstract operation cannot be used as a concrete implementation either.

### Offer a narrow operation to subclasses

**Protected access** permits access within the declaring package and, under Java’s subclass rules, from subclasses. A **package** groups related Java types under a name. This lesson uses a protected method from inside its subclass on the current object; cross-package access through other references has additional rules.

Keep stored state private when subclasses only need a narrow operation:

```java
class NamedItem {
    private String name;
    public NamedItem(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
}
class ItemHeading extends NamedItem {
    public ItemHeading(String name) {
        super(name);
    }
    public String heading() {
        return "Item: " + getName();
    }
}
ItemHeading item = new ItemHeading("Map");
System.out.println(item.heading());
```

The output is `Item: Map`. ItemHeading uses the inherited protected getName operation. It does not read or change name directly. Protected does not mean “subclasses only,” and it does not automatically make a field safe to expose. Here the private field plus a narrow getter keeps the stored value under the base class’s control.

### Combine the choices deliberately

The worked example uses an abstract NamedJob base with a private name, a constructor, a protected getter, an implemented description, and an abstract minutes operation. ReadingJob provides the missing calculation and extends the existing description.

| Mechanism | Purpose in the job design |
|---|---|
| extends NamedJob | Declare that the concrete job follows the base-class relationship. |
| super(name) | Initialize shared name state before the subclass stores its own count. |
| abstract int minutes() | Require each concrete kind of job to calculate its own duration. |
| @Override | Check the matching implementation or replacement method. |
| super.description() | Reuse the existing name-based description inside the override. |
| protected getName() | Let subclass code obtain the name through a defined operation. |
| NamedJob reference | Let a caller use base operations while the actual object supplies specialized behavior. |

A base reference only exposes operations declared by its type. A ReadingJob-only heading method is not available through a NamedJob variable. Keep a ReadingJob reference when the task specifically needs that extra method.

Use inheritance when the subclass can honor the base class’s meaning and operations. Sharing code is useful, but a misleading relationship makes callers harder to reason about. Keep this lesson’s hierarchy one level deep, and initialize fields directly in constructors without calling overridable methods from them.


## Video Demonstration

Follow the name from the subclass constructor into the abstract base. Then compare the inherited description, the override that extends it, and the required time calculation.

<video controls preload="metadata" width="960">
  <source src="media/02_inheritance_and_abstract_classes/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/02_inheritance_and_abstract_classes/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the inheritance and abstract classes demonstration transcript](media/02_inheritance_and_abstract_classes/transcript.md).


## Worked Example

**Subgoal 1: define shared state and promises.** NamedJob stores a private name and requires minutes.

**Subgoal 2: specialize the job.** ReadingJob passes the name to super, stores its page count, and supplies its calculation and description.

**Subgoal 3: choose the appropriate reference.** Use the ReadingJob reference for its heading and a NamedJob reference for the shared operations.


In [ ]:
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Guide", 3);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());


Expected output:

```text
Job: Guide
Guide reading
Minutes: 6
```

Both references point to the same ReadingJob object. Its inherited protected getter supplies Guide for the heading. Its description override calls the base description before adding reading. The minutes override returns 3 times 2. The abstract base supplies shared state and implemented behavior without being directly instantiated.


## Predict, Run, Trace, and Explain

### Predict a new reading job

Read the complete program before running it. Predict all three lines. Trace the name and page count through construction, then name the method body used by each printed call. For description, include the deliberate call to the base implementation. Record the declared type and actual class for both references before opening the answer.

My predicted three lines:

Where the name is stored:

Where the page count is stored:

Declared type and actual class of reading:

Declared type and actual class of job:

The method bodies used for heading, description and minutes:


In [ ]:
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Manual", 5);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());


Run the complete cell once. Keep your original prediction and compare every line. If a line differs, identify whether the difference came from constructor state, method selection or the explicit super.description() call. Run the full class definitions and caller for each attempt so the constructor inputs and instance state are recreated.

My original prediction:

My actual complete output:

Which lines match or differ:

The constructor or method step explaining a difference:

My corrected explanation after running:

### Separate construction from method selection

Trace the same ReadingJob from construction to the three calls. Complete both tables. Explain how super(name) differs from super.description(), and why neither creates another object. Identify the implemented base method, the abstract method requiring a subclass body, and the protected getter used by subclass code. Explain why the private name field is not directly accessible in ReadingJob. Decide whether job.heading() or new NamedJob("Manual") would be allowed; keep these proposed invalid operations in Markdown. Then compare the supplied QuickJob below, which does not override description.

| Construction step | Part initialized | Stored value |
|---|---|---|
| NamedJob constructor via super(name) | | |
| Remaining ReadingJob constructor | | |


| Caller expression | Declared reference type | Actual object class | Method body or bodies | Result |
|---|---|---|---|---|
| reading.heading() | | | | |
| job.description() | | | | |
| job.minutes() | | | | |


How super(name) differs from super.description():

Why there is one object and two references:

Which base method already has a body and which requires one:

Why getName() is usable inside ReadingJob but name is private:

Why protected does not mean subclasses only:

Why job.heading() and direct NamedJob construction are invalid:

How my trace explains the actual output:

<details>
<summary>Show answer</summary>

The new expression creates one ReadingJob. ReadingJob first calls super(name), so NamedJob stores Manual in its private name field. The subclass then stores 5 in pages. reading has declared type ReadingJob; job has declared type NamedJob, and both refer to that same object. reading.heading() uses the inherited protected getName() to obtain Manual. job.description() selects ReadingJob.description; its super.description() call deliberately uses NamedJob.description, which obtains the name, before the subclass adds reading. job.minutes() selects ReadingJob.minutes and returns 5 * 2 = 10. The abstract NamedJob type can declare the reference even though NamedJob cannot itself be constructed. NamedJob.description has a body; NamedJob.minutes is abstract and ends with a semicolon. ReadingJob supplies minutes and replaces description. Constructors are not inherited: ReadingJob declares its own constructor and calls the named base constructor. The protected getter is used inside the subclass on the current object while the underlying name stays private. Protected access also includes the declaring package; it does not mean subclasses only. job.heading() is unavailable through the NamedJob declared type, and direct construction of NamedJob is disallowed because the class is abstract.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Manual", 5);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Job: Manual
Manual reading
Minutes: 10
```

Common error: Treating super(name) as a second new object. Using the base description alone for the call on a ReadingJob object. Predicting the earlier Guide and 3 values instead of these constructor inputs.

</details>


### Inherit one method and implement another

Predict the two lines from this complete QuickJob program, then run it. QuickJob supplies minutes but does not declare description. Trace which body job.description() uses and compare it with ReadingJob.description. Count the new expressions and explain how an abstract base constructor still initializes the new concrete object. Before opening the answer, decide whether deleting QuickJob.minutes would leave this class complete enough to construct, and explain why. Keep that proposed deletion as a written diagnosis; do not run an incomplete class.

My predicted two lines:

My actual two lines:

Which description body runs and why:

Which minutes body runs and why:

Where the name Check is stored:

Number of constructed objects:

Why NamedJob’s constructor runs although the class is abstract:

Whether QuickJob would remain concrete without minutes:

How this differs from ReadingJob’s description override:


In [ ]:
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class QuickJob extends NamedJob {
    public QuickJob(String name) {
        super(name);
    }
    @Override
    public int minutes() {
        return 1;
    }
}
NamedJob job = new QuickJob("Check");
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

QuickJob extends NamedJob and calls super(name), so NamedJob initializes the private name as Check inside the one QuickJob object. QuickJob does not override description, so its public inherited NamedJob.description body runs and returns the name through getName(). QuickJob implements the abstract minutes operation with a return value of 1. A NamedJob reference can invoke both operations. Deleting that implementation would leave the declared concrete QuickJob without the required minutes body; it would not be a complete concrete implementation. An abstract class can have a constructor used during subclass construction even though it cannot be directly instantiated.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class QuickJob extends NamedJob {
    public QuickJob(String name) {
        super(name);
    }
    @Override
    public int minutes() {
        return 1;
    }
}
NamedJob job = new QuickJob("Check");
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Check
Minutes: 1
```

Common error: Assuming every inherited method must be rewritten in the subclass. Assuming an abstract class cannot contain a constructor or implemented method. Removing the abstract requirement because an unrelated method was inherited.

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete the subclass connections

The displayed draft is incomplete and for reading only. Copy it into the empty cell. Replace RELATION, BASE_INIT, BASE_DESCRIPTION and NAME_READER using `extends`, `super(name)`, `super.description()` and `getName()`, each once. Keep BASE_INIT first in the constructor and keep the rest of the program unchanged. Predict all three lines, run the completed program, and explain what each replacement connects. Explain why this class relationship uses extends and why heading obtains the private name through a protected method.

This sample is for repair:

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class SortingJob RELATION NamedJob {
    private int items;
    public SortingJob(String name, int items) {
        BASE_INIT;
        this.items = items;
    }
    @Override
    public String description() {
        return BASE_DESCRIPTION + " sorting";
    }
    @Override
    public int minutes() {
        return items * 3;
    }
    public String heading() {
        return "Job: " + NAME_READER;
    }
}
SortingJob sorting = new SortingJob("Parcel", 2);
NamedJob job = sorting;
System.out.println(sorting.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```


My four replacements:

My predicted complete output:

My actual complete output:

The state initialized by each constructor:

Why super(name) must appear first in this Java runtime:

What super.description() returns before the suffix is added:

Why heading uses getName():

My post-run explanation:

<details>
<summary>Show answer</summary>

RELATION is extends, BASE_INIT is super(name), BASE_DESCRIPTION is super.description(), and NAME_READER is getName(). SortingJob is a subclass of NamedJob. Its constructor first supplies Parcel to the base constructor, then stores 2 in items. The heading uses the inherited protected getter. The description override reuses the base’s Parcel result before adding sorting. The required minutes implementation returns 2 * 3 = 6. The NamedJob reference permits description and minutes, while the SortingJob reference permits its additional heading.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class SortingJob extends NamedJob {
    private int items;
    public SortingJob(String name, int items) {
        super(name);
        this.items = items;
    }
    @Override
    public String description() {
        return super.description() + " sorting";
    }
    @Override
    public int minutes() {
        return items * 3;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
SortingJob sorting = new SortingJob("Parcel", 2);
NamedJob job = sorting;
System.out.println(sorting.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Job: Parcel
Parcel sorting
Minutes: 6
```

Common error: Using implements for this class-to-class relationship. Passing no name to a base constructor that requires one. Calling description() again instead of the intended superclass body. Trying to use the private name field in the heading.

</details>


### Change an override while reusing the base result

Start with the complete Manual/5 prediction program. Change only ReadingJob.description so it returns `"Reading: " + super.description()`. Leave NamedJob, the constructors, heading, minutes and caller unchanged. Predict all three lines before running. Explain why only the description line changes and why the call through job still reaches the override. Then change only the constructor inputs to Outline and 0, predict and run the whole program again. Explain which results depend on the name and which depend on pages. Restore Manual and 5 when finished. Keep the explicit super call; do not replace it with a call to description() on the current object.


In [ ]:
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Manual", 5);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());


My changed return expression:

My predicted Manual/5 output:

My actual Manual/5 output:

Why only the description line changes:

Why job.description() selects the subclass body:

My predicted Outline/0 output:

My actual Outline/0 output:

Which results depend on the name and on pages:

My output after restoring Manual/5:

<details>
<summary>Show answer</summary>

The call job.description() still selects ReadingJob.description because the actual object is a ReadingJob. Inside that override, super.description() supplies the base result Manual; the new expression puts Reading: before it. The constructor state, heading and minutes implementation are unchanged, so the first and third lines remain Job: Manual and Minutes: 10. With Outline and 0, the shared name changes both text results while zero pages produce zero minutes. A plain description() call from inside this override would call the override again instead of selecting the base body.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return "Reading: " + super.description();
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Manual", 5);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Job: Manual
Reading: Manual
Minutes: 10
```

Common error: Changing the base description and affecting the wrong part of the design. Hard-coding Manual instead of using the base result. Replacing the explicit superclass call with another call to the override. Changing minutes even though the task changes only the description.

**Additional test: `Modified description with Outline and zero pages`.** The name travels through the base constructor and getter; the zero page count travels through the subclass constructor and calculation. The changed input also catches a hard-coded Manual description.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return "Reading: " + super.description();
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Outline", 0);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Job: Outline
Reading: Outline
Minutes: 0
```

</details>


### Repair the superclass constructor order

The displayed program uses the canonical Guide/3 inputs, but the subclass constructor places its superclass call too late for this Java 21 runtime. Do not run the draft as written. Identify the two constructor statements that must change order. Copy the complete program into the empty cell and move super(name) before the subclass field assignment. Preserve the statement contents and everything else. Predict the repaired output and run the complete repair. Explain why omitting super(name) instead would not supply the required base name, why constructors are not inherited, and why this call initializes the same object.

This sample is for repair:

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        this.pages = pages;
        super(name);
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Guide", 3);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```


The two statements in the faulty order:

My repaired order:

Why the base name must be initialized first:

Why deleting super(name) is not an equivalent repair:

My predicted three lines:

My actual three lines:

Evidence that the methods use the initialized state:

My post-run explanation of one object and two constructor bodies:

<details>
<summary>Show answer</summary>

The explicit super(name) call must be first in this Java 21 constructor. It initializes the NamedJob portion with Guide before ReadingJob stores 3 in pages. Moving the call, while keeping both statements, restores the original canonical program. Omitting it would not satisfy NamedJob’s constructor that requires a String argument. ReadingJob declares its own constructor; it does not inherit NamedJob’s constructor. The one new ReadingJob expression creates one object whose base and subclass portions are initialized in order. Its heading and description use Guide, and its minutes result is 3 * 2 = 6.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class ReadingJob extends NamedJob {
    private int pages;
    public ReadingJob(String name, int pages) {
        super(name);
        this.pages = pages;
    }
    @Override
    public String description() {
        return super.description() + " reading";
    }
    @Override
    public int minutes() {
        return pages * 2;
    }
    public String heading() {
        return "Job: " + getName();
    }
}
ReadingJob reading = new ReadingJob("Guide", 3);
NamedJob job = reading;
System.out.println(reading.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Job: Guide
Guide reading
Minutes: 6
```

Common error: Deleting the superclass call rather than placing it correctly. Moving the field assignment into the base class and changing the contract. Making the private base name public as an unrelated workaround. Claiming super(name) constructs a second object.

</details>


## Independent Practice

### Build a concrete job from the abstract base

Reuse the supplied NamedJob abstract base and create BuildJob with a private steps field, super(name), a minutes implementation of steps * 5, a description that appends " build" to super.description(), and a heading that returns "Task: " plus protected getName(). Construct BuildJob("Lab",4), assign it to NamedJob job, and print the heading through the BuildJob reference followed by the description and labeled minutes through the base reference. Test zero steps. Explain why NamedJob cannot be directly instantiated and why job.heading() is not available through that declared type. Inputs have nonnegative page/step counts. Copy the supplied complete NamedJob base into the work cell, then add your BuildJob and full caller. Preserve the base exactly. Use @Override on description and minutes, keep super(name) first in the constructor, and use the protected getter from the subclass heading. Predict all three lines before running. Explain which methods are implemented, inherited or overridden and why each call uses the stated reference.

**Provided starter code.** Keep this definition with your own implementation:

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
```

My constructor and inheritance choices:

My predicted three lines:

My actual three lines:

How the name reaches the base field:

How the step count reaches the subclass field:

Which body each printed call reaches:

Why NamedJob construction and job.heading() are not valid alternatives:

How protected getName() supports the subclass without exposing name:

My post-run explanation and any correction:


### Test zero steps and changed constructor inputs

Test every row with the complete unchanged NamedJob and BuildJob definitions. Change only the constructor arguments in the caller and run the whole program so each case starts with a new BuildJob. Predict all three lines before each run and record the actual lines. For zero steps, explain why the name-based lines remain while minutes becomes zero. Use one step to check the per-step rule. Change both the name and count to Annex and 2 to check that the inherited and overridden methods use current constructor state. Keep the heading call on building and the description/minutes calls on job. Explain why neither direct NamedJob construction nor job.heading() is a valid replacement; leave those invalid alternatives in Markdown. Repair any mismatch, repeat the cases and restore Lab/4. Do not add negative-input behavior that the contract does not specify.

| Name / steps | Predicted three lines | Actual three lines | Match or repair |
|---|---|---|---|
| Lab / 4 | | | |
| Lab / 0 | | | |
| Lab / 1 | | | |
| Annex / 2 | | | |


Why zero steps still leaves a named object:

What the one-step case checks:

Why both text lines change for Annex:

Which body supplies each result through each declared reference:

Why direct NamedJob construction and job.heading() stay invalid:

My correction and cases repeated:

My actual output after restoring Lab/4:


<details>
<summary>Show answer</summary>

BuildJob extends NamedJob. Its constructor first passes Lab to the superclass constructor, which stores the private name, then stores 4 in the private steps field. Its minutes implementation supplies the abstract requirement and returns 4 * 5 = 20. Its description override calls super.description(), receives Lab, and appends build. Its heading uses the inherited protected getName() to build Task: Lab. building and job refer to the same BuildJob object. The BuildJob reference exposes heading; the NamedJob reference exposes description and minutes, whose calls select the BuildJob implementations. NamedJob cannot be directly constructed because it is abstract, and job.heading() is unavailable through its declared type. Counts are nonnegative classroom integers with products fitting int; these implementations rely on that input rule and do not validate it. For Lab/0, the constructors still create a valid concrete object with the name Lab; the two text lines stay Task: Lab and Lab build, while steps * 5 gives zero. Lab/1 yields five minutes and checks the single-step rule. Annex/2 must change both name-based lines and yield ten minutes. All methods still act on one BuildJob through the appropriate declared reference. These tests separate shared name state from subclass numeric state and catch hard-coded results. They do not make NamedJob constructible or add heading to the NamedJob declared type. Recreate the complete object for each case and restore the baseline afterward.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class BuildJob extends NamedJob {
    private int steps;
    public BuildJob(String name, int steps) {
        super(name);
        this.steps = steps;
    }
    @Override
    public String description() {
        return super.description() + " build";
    }
    @Override
    public int minutes() {
        return steps * 5;
    }
    public String heading() {
        return "Task: " + getName();
    }
}
BuildJob building = new BuildJob("Lab", 4);
NamedJob job = building;
System.out.println(building.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Task: Lab
Lab build
Minutes: 20
```

Common error: Changing the supplied abstract base instead of implementing its requirement. Reading the private name directly in BuildJob. Giving minutes a changed method name, parameter list, or result type or omitting its body. Calling heading through the NamedJob reference. Hard-coding the name or step count instead of using constructor state.

**Additional test: Lab with zero steps.** The name is initialized normally and both text operations still work. Zero is a valid step count and produces zero minutes.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class BuildJob extends NamedJob {
    private int steps;
    public BuildJob(String name, int steps) {
        super(name);
        this.steps = steps;
    }
    @Override
    public String description() {
        return super.description() + " build";
    }
    @Override
    public int minutes() {
        return steps * 5;
    }
    public String heading() {
        return "Task: " + getName();
    }
}
BuildJob building = new BuildJob("Lab", 0);
NamedJob job = building;
System.out.println(building.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Task: Lab
Lab build
Minutes: 0
```

**Additional test: Lab with one step.** One step produces five minutes while the shared name-based behavior remains unchanged.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class BuildJob extends NamedJob {
    private int steps;
    public BuildJob(String name, int steps) {
        super(name);
        this.steps = steps;
    }
    @Override
    public String description() {
        return super.description() + " build";
    }
    @Override
    public int minutes() {
        return steps * 5;
    }
    public String heading() {
        return "Task: " + getName();
    }
}
BuildJob building = new BuildJob("Lab", 1);
NamedJob job = building;
System.out.println(building.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Task: Lab
Lab build
Minutes: 5
```

**Additional test: Annex with two steps.** Both heading and description must use Annex from the base state, and the subclass must compute 2 * 5 rather than a hard-coded baseline result.

```java
abstract class NamedJob {
    private String name;
    public NamedJob(String name) {
        this.name = name;
    }
    protected String getName() {
        return name;
    }
    public String description() {
        return getName();
    }
    public abstract int minutes();
}
class BuildJob extends NamedJob {
    private int steps;
    public BuildJob(String name, int steps) {
        super(name);
        this.steps = steps;
    }
    @Override
    public String description() {
        return super.description() + " build";
    }
    @Override
    public int minutes() {
        return steps * 5;
    }
    public String heading() {
        return "Task: " + getName();
    }
}
BuildJob building = new BuildJob("Annex", 2);
NamedJob job = building;
System.out.println(building.heading());
System.out.println(job.description());
System.out.println("Minutes: " + job.minutes());
```

Expected output:

```text
Task: Annex
Annex build
Minutes: 10
```

</details>


## Summary

A subclass extends one superclass. Its constructor initializes the base portion through super, and it may inherit or override accessible instance methods. super.method() deliberately calls a base implementation. An abstract class cannot be constructed directly and may require concrete subclasses to implement abstract methods. Protected operations can support subclass work while fields remain private.

Close the answers. Explain the difference between super(name) and super.description(), then trace a call through a base-typed reference.


## Reflection

Choose two kinds of work in a real setting that share a useful name or description but calculate effort differently. Propose one abstract operation and one implemented base operation. Explain why both concrete kinds can honor the same promise.

**My design and explanation:**

Next, you will make class and interface definitions reusable across different element types with generics.


## Supplemental Reading

- [Java class inheritance](https://dev.java/learn/inheritance/what-is-inheritance/) explains superclass and subclass relationships.
- [Abstract methods and classes](https://dev.java/learn/inheritance/abstract-classes/) explains incomplete base definitions and concrete subclasses.
- [Java 21 class rules](https://docs.oracle.com/javase/specs/jls/se21/html/jls-8.html) specify overriding and constructor calls.
- [Java 21 access control](https://docs.oracle.com/javase/specs/jls/se21/html/jls-6.html#jls-6.6) defines public, private, package, and protected access.
